# 04 — De predicción a recomendación

Requiere haber ejecutado `02_pipeline_nowcast.ipynb` (carga su modelo entrenado). Opcionalmente usa el modelo de `03_forecasting.ipynb` para la variante de recomendación anticipativa.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from scipy.spatial import cKDTree
import pickle
import joblib
import itertools
import time

plt.style.use('ggplot')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## Cargar configuración y modelos ya entrenados

No reentrenamos nada aquí -- cargamos lo que `02_pipeline_nowcast.ipynb` (y opcionalmente `03_forecasting.ipynb`) ya guardaron.

In [ ]:
with open('../artefactos/configuracion.pkl', 'rb') as f:
    configuracion = pickle.load(f)

VARIABLE_CALIDAD = configuracion['VARIABLE_CALIDAD']
VARIABLES_XMV = configuracion['VARIABLES_XMV']
RUTA_CSV = configuracion['RUTA_CSV']
COLUMNA_TIEMPO = configuracion['COLUMNA_TIEMPO']

df_raw = pd.read_csv(RUTA_CSV)
df_raw = df_raw.sort_values(COLUMNA_TIEMPO).reset_index(drop=True)

modelo_nowcast = joblib.load('../artefactos/modelo_nowcast_una_sim.joblib')
df_feat = pd.read_csv('../artefactos/df_feat_nowcast.csv')
with open('../artefactos/cols_feat_nowcast.pkl', 'rb') as f:
    cols_feat = pickle.load(f)

# Empaquetamos en la misma forma que evaluar_modelo() devolvía, para reutilizar el
# resto del código de recomendación sin cambios
res_calidad = {'modelo': modelo_nowcast}

# El modelo de forecasting es opcional -- solo si vas a usar la recomendación anticipativa
try:
    modelo_forecast = joblib.load('../artefactos/modelo_forecast_una_sim.joblib')
    df_feat_fc = pd.read_csv('../artefactos/df_feat_forecast.csv')
    with open('../artefactos/cols_feat_forecast.pkl', 'rb') as f:
        cols_feat_fc = pickle.load(f)
    res_forecast = {'modelo': modelo_forecast}
    HORIZONTE_FORECAST = 5
    print("Modelo de forecasting también cargado (disponible para recomendación anticipativa)")
except FileNotFoundError:
    print("Modelo de forecasting no encontrado -- ejecuta 03_forecasting.ipynb si quieres la variante anticipativa")

print("Modelo de nowcast cargado. VARIABLE_CALIDAD:", VARIABLE_CALIDAD)

## 8. De predicción a recomendación

Aquí está la pieza nueva. La idea:

1. Tomamos el **estado actual** del proceso (última fila de datos: lags, rolling, etc. ya calculados salvo las XMV, que son lo que queremos decidir).
2. Probamos una **rejilla de combinaciones posibles de XMV**, dentro del rango que el proceso ya ha mostrado en los datos históricos (nunca fuera — el árbol no sabe extrapolar).
3. Para cada combinación, reconstruimos el vector de features y le pedimos al modelo que prediga la calidad resultante.
4. Nos quedamos con la combinación que maximiza (o minimiza, según el objetivo) la calidad predicha.

### La advertencia que de verdad importa aquí

Esto **no es optimización real de proceso** — es "probar muchas combinaciones ya vistas y quedarte con la mejor según lo que el modelo cree". Si el óptimo real está fuera del rango histórico, este método nunca lo va a encontrar, y si lo fuerzas a buscar fuera de rango, el modelo va a inventarse una predicción sin ninguna base real. Por eso la rejilla está anclada estrictamente al min/max observado — es una salvaguarda, no un capricho.

In [ ]:
def construir_vector_estado_actual(df_raw, df_feat, cols_feat, variables_entrada, idx=-1):
    """
    Toma la última fila disponible de df_feat (con todos los lags/rolling/diff ya calculados)
    como snapshot del estado del proceso, para usarla como plantilla en la búsqueda.
    """
    return df_feat[cols_feat].iloc[[idx]].copy()


def recomendar_ajuste(modelo, df_raw, df_feat, cols_feat, variables_xmv, variable_calidad,
                       modo='rejilla', n_puntos_por_variable=5, n_muestras_aleatorias=2000,
                       objetivo='maximizar', seed=42, estado_base=None):
    """
    Busca la mejor combinación de XMV, ACOTADA al rango histórico observado (nunca extrapola).

    modo='rejilla': malla completa (itertools.product). Exhaustivo pero crece como
                     n_puntos_por_variable ** n_variables -- inviable con muchas variables
                     (con 11 XMV y 5 puntos/variable son ~49 millones de combinaciones).
                     Útil solo con pocas variables (hasta 4-5) o pocos puntos por variable.

    modo='aleatorio': muestrea n_muestras_aleatorias combinaciones al azar dentro del rango
                       válido de cada variable. El coste NO depende del número de variables,
                       solo del número de muestras que decidas pagar -- por eso escala a
                       cualquier número de XMV sin explotar en memoria ni en tiempo.
                       En espacios de muchas dimensiones, random search suele acercarse al
                       óptimo de grid search con una fracción ínfima del coste, porque la
                       mayoría de combinaciones de una malla completa son redundantes.

    estado_base: si se pasa (un DataFrame de 1 fila con las columnas cols_feat), se usa como
                 punto de partida en vez del último dato observado -- por ejemplo, el estado
                 PROYECTADO a h pasos vista con el modelo de forecasting (sección 6), para una
                 recomendación anticipativa en vez de puramente reactiva.

    IMPORTANTE: en ambos modos, todas las combinaciones se construyen de golpe en un único
    DataFrame y se evalúan con UNA sola llamada a modelo.predict() (vectorizado), no una
    llamada por combinación -- eso es lo que evita el overhead fijo de predict() multiplicado
    por miles/millones de veces, que es la causa más común de lentitud aquí (no el tamaño
    del modelo, ni la falta de GPU).
    """
    if estado_base is None:
        estado_base = construir_vector_estado_actual(df_raw, df_feat, cols_feat, variables_xmv)
    rng = np.random.default_rng(seed)

    rangos = {var: (df_raw[var].min(), df_raw[var].max()) for var in variables_xmv}

    if modo == 'rejilla':
        rejillas = {var: np.linspace(vmin, vmax, n_puntos_por_variable)
                    for var, (vmin, vmax) in rangos.items()}
        combinaciones = np.array(list(itertools.product(*[rejillas[v] for v in variables_xmv])))
    elif modo == 'aleatorio':
        combinaciones = np.column_stack([
            rng.uniform(rangos[var][0], rangos[var][1], n_muestras_aleatorias)
            for var in variables_xmv
        ])
    else:
        raise ValueError("modo debe ser 'rejilla' o 'aleatorio'")

    n_combos = len(combinaciones)

    # Construir TODAS las filas de una vez (vectorizado), en vez de una por iteración
    filas = pd.concat([estado_base] * n_combos, ignore_index=True)
    for j, var in enumerate(variables_xmv):
        if var in filas.columns:
            filas[var] = combinaciones[:, j]

    # UNA sola llamada a predict() para todas las combinaciones
    preds = modelo.predict(filas[cols_feat])

    df_resultados = pd.DataFrame(combinaciones, columns=variables_xmv)
    df_resultados['calidad_predicha'] = preds

    ascending = (objetivo == 'minimizar')
    df_resultados = df_resultados.sort_values('calidad_predicha', ascending=ascending).reset_index(drop=True)
    return df_resultados

### Ejecutar la recomendación

Buscamos la combinación de XMV que **maximiza** la calidad predicha (cambia `objetivo='minimizar'` si tu variable de calidad es del tipo "cuanto menos, mejor" — como sería el caso de acidez).

In [ ]:
# Con 11 variables XMV reales, modo='rejilla' con 5 puntos/variable son ~48.8 millones
# de combinaciones -- eso es lo que te mató el kernel. Con >= 5-6 variables, usa
# modo='aleatorio': el coste no depende de cuántas XMV tengas, solo de cuántas
# muestras decidas pagar (ver benchmark en la siguiente celda para comparar ambos
# modos con un tamaño de rejilla que SÍ es seguro, y ver por qué el real no lo era).
df_recomendaciones = recomendar_ajuste(
    modelo=res_calidad['modelo'],
    df_raw=df_raw,
    df_feat=df_feat,
    cols_feat=cols_feat,
    variables_xmv=VARIABLES_XMV,
    variable_calidad=VARIABLE_CALIDAD,
    modo='aleatorio',
    n_muestras_aleatorias=2000,
    objetivo='maximizar'
)

print("1 combinaciones recomendadas:")
df_recomendaciones.head(1)

### Por qué esto ya no "tarda una barbaridad": rejilla vs aleatorio, medido

Dos cosas cambiaron respecto a la primera versión: (1) `predict()` se llama **una sola vez** con todas las combinaciones juntas en vez de una vez por combinación, y (2) puedes elegir `modo='aleatorio'` para que el coste no explote con el número de variables. Vamos a medirlo con tus propios datos y modelo, no de oídas.

In [ ]:
import time

n_xmv = len(VARIABLES_XMV)
# Con muchas variables, n_puntos_por_variable pequeño para que la rejilla no reviente
# (5 puntos ^ 11 variables = 48.8M combinaciones -- eso es justo lo que mató el kernel arriba).
n_puntos_seguro = 2 if n_xmv >= 6 else 5

for modo, kwargs in [
    ('rejilla', {'n_puntos_por_variable': n_puntos_seguro}),
    ('aleatorio', {'n_muestras_aleatorias': 500}),
]:
    t0 = time.time()
    df_bench = recomendar_ajuste(
        modelo=res_calidad['modelo'], df_raw=df_raw, df_feat=df_feat, cols_feat=cols_feat,
        variables_xmv=VARIABLES_XMV, variable_calidad=VARIABLE_CALIDAD, modo=modo, objetivo='maximizar',
        **kwargs
    )
    t1 = time.time()
    mejor_calidad = df_bench.iloc[0]["calidad_predicha"]
    print(f"modo={modo:10s}  combinaciones={len(df_bench):8d}  tiempo={t1-t0:.4f}s  mejor_calidad={mejor_calidad:.4f}")

print()
print(f"Con tus {n_xmv} variables XMV reales:")
print(f"  modo=rejilla con 5 puntos/variable    -> {5**n_xmv:,} combinaciones  <- esto te mató el kernel arriba")
print(f"  modo=aleatorio con 2000 muestras       -> 2,000 combinaciones (mismo coste siempre, no depende de n_xmv)")

### Comparación: recomendación reactiva vs anticipativa

Usamos el estado proyectado (sección 6) como punto de partida y comparamos la recomendación resultante contra la que ya tenías (partiendo del último dato observado). Si el proceso está razonablemente estable, ambas deberían parecerse bastante -- una diferencia grande es la señal de que anticiparte al futuro sí está cambiando la decisión, no solo maquillándola.

In [ ]:
# OJO: aquí había un bug de diseño en el notebook original -- se pasaba
# df_feat_fc[cols_feat_fc] (las columnas del modelo de FORECASTING) como estado_base a
# recomendar_ajuste(modelo=res_calidad, cols_feat=cols_feat, ...), que espera las
# columnas del modelo de NOWCAST. Con configuraciones de features distintas entre ambos
# modelos (nowcast y forecasting no comparten n_lags/rolling), esas columnas no coinciden
# y la llamada falla con KeyError -- lo comprobamos ejecutando de principio a fin.
#
# El modelo de forecasting predice un ÚNICO escalar (la calidad proyectada), no el vector
# completo de features -- no hay una forma directa de "proyectar" las 150+ columnas del
# modelo de nowcast a partir de eso. Lo correcto es usar el ESTADO ACTUAL (última fila de
# df_feat, columnas correctas) como base, y usar calidad_proyectada (ya calculada arriba,
# sección de "estado proyectado") solo como CONTEXTO informativo -- para saber si merece
# la pena mirar la recomendación anticipativa, no como vector de entrada al modelo.
estado_base_actual = df_feat[cols_feat].iloc[[-1]].copy()

df_recomendaciones_anticipativa = recomendar_ajuste(
    modelo=res_calidad['modelo'],
    df_raw=df_raw,
    df_feat=df_feat,
    cols_feat=cols_feat,
    variables_xmv=VARIABLES_XMV,
    variable_calidad=VARIABLE_CALIDAD,
    modo='aleatorio',
    n_muestras_aleatorias=2000,
    objetivo='maximizar',
    estado_base=estado_base_actual
)

try:
    with open('../artefactos/proyeccion_calidad.pkl', 'rb') as f:
        proyeccion = pickle.load(f)
    print(f"(Contexto: calidad_proyectada a +{HORIZONTE_FORECAST} pasos sin intervención = {proyeccion['calidad_proyectada']:.4f}, calidad_actual = {proyeccion['calidad_actual']:.4f})")
except FileNotFoundError:
    print("(No se encontró la proyección de calidad -- ejecuta 03_forecasting.ipynb completo si la necesitas)")

print("Recomendación reactiva (estado actual):")
print(df_recomendaciones.iloc[0][VARIABLES_XMV + ['calidad_predicha']])
print()
print(f"Recomendación anticipativa (estado proyectado a +{HORIZONTE_FORECAST} pasos):")
print(df_recomendaciones_anticipativa.iloc[0][VARIABLES_XMV + ['calidad_predicha']])

## 9. Comprobación de fiabilidad — ¿la recomendación está dentro de lo que el modelo realmente conoce?

Esta celda es la que convierte "una recomendación" en "una recomendación en la que puedes confiar un poco más". Comprobamos si la mejor combinación encontrada corresponde a una zona del espacio de XMV donde hubo **suficientes datos históricos parecidos** — si la mejor recomendación vive en una esquina del espacio con pocos puntos de entrenamiento cerca, es una señal de alerta, no una señal de éxito.

In [ ]:
from scipy.spatial import cKDTree

def distancia_a_datos_historicos(mejor_combo, df_raw, variables_xmv):
    """
    Distancia (normalizada) de la combinación recomendada al punto histórico más cercano,
    usando un KDTree en vez de una matriz de distancias completa (cdist).

    Con un histórico grande (aquí, ~250.000 filas si df_raw son las 500 simulaciones
    completas), calcular todas las distancias contra todas las filas no es necesario --
    solo buscamos EL vecino más cercano, que es justo lo que un KDTree resuelve sin
    construir esa matriz gigante en memoria.
    """
    datos_hist = df_raw[variables_xmv].values
    rangos = datos_hist.max(axis=0) - datos_hist.min(axis=0)
    rangos[rangos == 0] = 1
    datos_hist_norm = datos_hist / rangos
    combo_norm = np.array(mejor_combo) / rangos

    tree = cKDTree(datos_hist_norm)
    dist_min, _ = tree.query(combo_norm)
    return dist_min

mejor_combo = df_recomendaciones.iloc[0][VARIABLES_XMV].values
dist_min = distancia_a_datos_historicos(mejor_combo, df_raw, VARIABLES_XMV)

# Referencia: distancia típica entre puntos históricos consecutivos, para dar contexto a la cifra.
# Aquí NO usamos cdist (que compararía todas las filas contra todas -> misma explosión de
# memoria que arriba). Solo queremos la distancia fila[i] vs fila[i+1], una comparación
# uno-a-uno, así que una resta vectorizada basta y es muchísimo más barata.
datos_hist_norm_ref = df_raw[VARIABLES_XMV].values / (df_raw[VARIABLES_XMV].max() - df_raw[VARIABLES_XMV].min()).values
dist_referencia = np.median(np.linalg.norm(datos_hist_norm_ref[1:] - datos_hist_norm_ref[:-1], axis=1))

print(f"Mejor combinación recomendada: {dict(zip(VARIABLES_XMV, mejor_combo))}")
calidad_predicha_top = df_recomendaciones.iloc[0]["calidad_predicha"]
print(f"Calidad predicha: {calidad_predicha_top:.4f}")
print(f"Distancia (normalizada) al dato histórico más cercano: {dist_min:.4f}")
print(f"Distancia típica de referencia entre muestras consecutivas: {dist_referencia:.4f}")

if dist_min > 3 * dist_referencia:
    print("\n⚠️  ALERTA: esta recomendación está lejos de cualquier combinación vista en los datos.")
    print("   El modelo está extrapolando -- trátala como hipótesis a validar, no como instrucción a aplicar.")
else:
    print("\n✓ La recomendación está razonablemente cerca de zonas ya observadas en los datos históricos.")

## 10. Qué falta para que esto sea un sistema de recomendación real (no un ejercicio)

Deja esto anotado para cuando tengas los datos reales de la almazara, porque son las piezas que este notebook simplifica a propósito para que puedas centrarte en la arquitectura:

- **La variable de calidad real puede llegar con retardo y a otra cadencia** (análisis de laboratorio vs sensores en línea). Aquí la calidad se genera al mismo ritmo que el resto — en tu caso probablemente no será así, y tendrás que resolver un forecasting de horizonte largo antes de poder recomendar nada.
- **La búsqueda en rejilla no escala bien con muchas variables manipulables.** Con 3 XMV y 5 puntos por variable ya son 125 combinaciones; con 11 XMV (como en el TEP real) serían millones. Para eso existen métodos de optimización más eficientes (optimización bayesiana, algoritmos genéticos) — pero no los necesitas para aprender el concepto.
- **El modelo no sabe nada de restricciones de seguridad o de proceso** (p. ej. "la presión no puede superar X"). Cualquier recomendación real necesita una capa de reglas duras por encima del modelo, no solo el óptimo estadístico.
- **La comprobación de distancia a datos históricos es una heurística simple**, no una medida de incertidumbre real del modelo. Para algo más riguroso, XGBoost puede combinarse con estimación de incertidumbre (ensembles, quantile regression) — es un paso natural una vez domines esta versión básica.
- **Sobre GPU**: acelera el *entrenamiento* de XGBoost (`tree_method='hist', device='cuda'`) sobre datasets grandes, y `predict()` vectorizado sobre muchas filas a la vez. No ayuda si el cuello de botella es cómo está escrito el bucle de búsqueda (llamadas individuales a predict, o generar una malla que no cabe en memoria) — eso se arregla vectorizando y con `modo='aleatorio'`, como en este notebook, no añadiendo hardware. Si algún día entrenas con millones de filas y el *entrenamiento* (no la recomendación) es lo lento, ahí sí GPU tiene sentido.
- **Si más adelante necesitas exprimir más la búsqueda que random search**, el siguiente escalón es optimización bayesiana (p. ej. librería `scikit-optimize` o `optuna`): en vez de muestrear a ciegas, usa los resultados ya evaluados para decidir qué combinación probar después, así que converge a un buen óptimo con muchas menos llamadas al modelo que random search. No lo necesitas todavía con pocas variables, pero es la pieza que te faltaría si el espacio de búsqueda creciera mucho más o cada evaluación del modelo fuera cara.